# HBS Case Study
## HW2: ProShares Hedge Replication ETF
### Group A


# Analyzing the Data

Use the data in `proshares_analysis_data.xlsx`. 

It has monthly data on financial indexes and ETFs from `Aug 2011` through `May 2025`.

## 1. 

For the series in the "hedge fund series" tab, report the following summary statistics:
* mean
* volatility
* Sharpe ratio

Annualize these statistics.

In [1]:
import pandas as pd 
path = "data/proshares_analysis_data.xlsx"
hedge_fund_series = pd.read_excel(path, sheet_name="hedge_fund_series", index_col=0)

In [2]:
import numpy as np
ann_mean = hedge_fund_series.mean() * 12
ann_vol = hedge_fund_series.std() * np.sqrt(12)
sharpe = ann_mean / ann_vol
summary_stats = pd.DataFrame({"annualized_mean": ann_mean, "annualized volatility": ann_vol, "sharpe ratio": sharpe})
print(summary_stats)

                annualized_mean  annualized volatility  sharpe ratio
HFRIFWI Index          0.051279               0.058796      0.872151
MLEIFCTR Index         0.038535               0.055240      0.697589
MLEIFCTX Index         0.036526               0.055102      0.662880
HDG US Equity          0.026880               0.057388      0.468396
QAI US Equity          0.028811               0.049823      0.578275


## 2.

For the series in the "hedge fund series" tab, calculate the following statistics related to tail-risk.
* Skewness
* Excess Kurtosis (in excess of 3)
* VaR (.05) - the fifth quantile of historic returns
* CVaR (.05) - the mean of the returns at or below the fifth quantile
* Maximum drawdown - include the dates of the max/min/recovery within the max drawdown period.

There is no need to annualize any of these statistics.

In [ ]:
skewness = hedge_fund_series.skew()
kurtosis = hedge_fund_series.kurtosis()
var_fifth = hedge_fund_series.quantile(0.05)
cvar_fifth = hedge_fund_series[hedge_fund_series <= var_fifth].mean()
#max drawdown = largest peak to trough 
cum_rets = (1 + hedge_fund_series).cumprod()
rolling_max = cum_rets.cummax()
drawdown = (cum_rets - rolling_max) / rolling_max
mdd = drawdown.min()

min_dates = {}
max_dates = {}
recovery_dates = {}

for col in hedge_fund_series.columns:
    min_date = drawdown[col].idxmin()
    max_date = cum_rets.loc[:min_date, col].idxmax()
    post_min = cum_rets.loc[min_date:, col]
    recovery_date = post_min[post_min >= cum_rets.loc[max_date, col]]
    min_dates[col] = min_date
    max_dates[col] = max_date
    recovery_dates[col] = recovery_date.index[0].date()

tail_summary = pd.DataFrame({
    "skewness" : skewness, "kurtosis": kurtosis, 
    "VaR (0.05)": var_fifth, "cVaR": cvar_fifth,
    "max drawdown": mdd, "date at trough": min_dates, "date at peak": max_dates,
    "date at recovery": recovery_dates
})
print(tail_summary)

                skewness  kurtosis  VaR (0.05)      cVaR  max drawdown  \
HFRIFWI Index  -0.948272  5.657351   -0.024008 -0.035992     -0.115473   
MLEIFCTR Index -0.289982  1.630917   -0.027002 -0.034993     -0.124302   
MLEIFCTX Index -0.273477  1.589790   -0.026970 -0.034937     -0.124388   
HDG US Equity  -0.274904  1.776481   -0.029919 -0.036831     -0.140720   
QAI US Equity  -0.433508  1.449184   -0.017171 -0.030995     -0.137716   

               date at trough date at peak date at recovery  
HFRIFWI Index      2020-03-31   2019-12-31       2020-08-31  
MLEIFCTR Index     2022-09-30   2021-06-30       2024-02-29  
MLEIFCTX Index     2022-09-30   2021-06-30       2024-02-29  
HDG US Equity      2022-09-30   2021-06-30       2024-07-31  
QAI US Equity      2022-09-30   2021-06-30       2024-02-29  


## 3. 

For the series in the "hedge fund series" tab, run a regression of each against SPY (found in the "merrill factors" tab.) Include an intercept. Report the following regression-based statistics:
* Market Beta
* Treynor Ratio
* Information ratio

Annualize these three statistics as appropriate.

In [52]:
import statsmodels.api as sm

merrill_factors = pd.read_excel(path, sheet_name="merrill_factors", index_col=0)
SPY = merrill_factors["SPY US Equity"]

rf = merrill_factors["USGG3M Index"].mean() *12 #risk-free rate

results = []

for col in hedge_fund_series.columns:
    y = hedge_fund_series[col].dropna()
    X = sm.add_constant(SPY.loc[y.index])
    reg = sm.OLS(y, X).fit()

    alpha = reg.params.iloc[0]
    beta = reg.params.iloc[1]

    treynor = ((y.mean() * 12) - rf)/beta

    info = (alpha / reg.resid.std()) * np.sqrt(12)

    results.append({"Series": col, "Beta": round(beta, 4), "Treynor Ratio (ann.)" : treynor, "Info Ratio (ann.)": info})

results = pd.DataFrame(results).set_index("Series")

display(results)


,Beta,Treynor Ratio (ann.),Info Ratio (ann.)
Series,,,
HFRIFWI Index,0.3463,0.104507,0.055336
MLEIFCTR Index,0.3425,0.068462,-0.436440
MLEIFCTX Index,0.3415,0.062782,-0.510548
HDG US Equity,0.3506,0.033637,-0.862386
QAI US Equity,0.3014,0.045532,-0.597796


## 4. 

Discuss the previous statistics, and what they tell us about...

* the differences between SPY and the hedge-fund series?
* which performs better between HDG and QAI.
* whether HDG and the ML series capture the most notable properties of HFRI.

In [103]:

def summary(r):
    return pd.Series({
        "ann_mean": r.mean()*12,
        "ann_vol":  r.std()*np.sqrt(12),
        "sharpe":   (r.mean()*12)/(r.std()*np.sqrt(12)),
        "skewness": r.skew(),
        "kurtosis": r.kurtosis(),
    })

print("SPY Stats\n" + "-"*22)
print(summary(SPY))

SPY Stats
----------------------
ann_mean    0.144741
ann_vol     0.143396
sharpe      1.009380
skewness   -0.412560
kurtosis    0.704481
dtype: float64


*Looking at the betas, all of the indices and ETFs have somewhere in the neighborhood of 30-35% exposure to S&P500. They all have a lower mean than SPY but their annualized vols are much lower than having direct exposure to market risk.*

*QAI outperforms HDG across our stats. It has higher mean return, lower vol, and thus better Sharpe. Additionally QAI has better Treynor and Info ratios than HDG.*

*While HDG and ML replicate the risk exposure of HFRI (they have similar betas i.e. market exposure AND similar volatilities to HFRI) they still fail to capture the composite's return (HFRI: 5.8% vs 3.9% and 2.7% for ML and HDG respectively).*


## 5. 

Report the correlation matrix for these assets.
* Show the correlations as a heat map.
* Which series have the highest and lowest correlations?

## 6.

Replicate HFRI with the six factors listed on the "merrill factors" tab. Include a constant, and run the unrestricted regression,

$$\begin{aligned}
r^{\text{hfri}}_{t} &= \alpha^{\text{merr}} + x_{t}^{\text{merr}}\beta^{\text{merr}} + \epsilon_{t}^{\text{merr}}\\[5pt]
\hat{r}^{\text{hfri}}_{t} &= \hat{\alpha}^{\text{merr}} + x_{t}^{\text{merr}}\hat{\beta}^{\text{merr}}
\end{aligned}$$

Note that the second equation is just our notation for the fitted replication.

a. Report the intercept and betas.

b. Are the betas realistic position sizes, or do they require huge long-short positions?

c. Report the R-squared.

d. Report the volatility of $\epsilon^{\text{merr}}$, the tracking error.

## 7.

Let's examine the replication out-of-sample (OOS).

Starting with $t = 61$ month of the sample, do the following:

* Use the previous 60 months of data to estimate the regression equation. 
This gives time-t estimates of the regression parameters, $\tilde{\alpha}^{\text{merr}}_{t}$ and $\tilde{\beta}^{\text{merr}}_{t}$.

* Use the estimated regression parameters, along with the time-t regressor values, $x^{\text{merr}}_{t}$, calculate the time-t replication value that is, with respect to the regression estimate, built "out-of-sample" (OOS).

$$\hat{r}^{\text{hfri}}_{t} \equiv \tilde{\alpha}^{\text{merr}} + (x_{t}^{\text{merr}})'\tilde{\beta}^{\text{merr}}$$

* Step forward to $t = 62$, and now use $t = 2$ through $t = 61$ for the estimation. Re-run the steps above, and continue this process throughout the data series. Thus, we are running a rolling, 60-month regression for each point-in-time.

How well does the out-of-sample replication perform with respect to the target?

***

# Exercise: Unconstrained Optimization

## Data

All the analysis below applies to the data set,
* `data/spx_returns_weekly.xlsx`
* The file has **weekly** returns.
* For annualization, use 52 periods per year.

Consider only the following 10 stocks...

In [27]:
TICKS =  ['AAPL','NVDA','MSFT','GOOGL','AMZN','META','TSLA','AVGO','BRK/B','LLY']

As well as the ETF,

In [28]:
TICK_ETF = 'SPY'

### Data Processing

In [29]:
INFILE = 'data/spx_returns_weekly.xlsx'
SHEET_INFO = 'spx names'
SHEET_RETURNS = 'spx returns'
SHEET_BENCH = 'additional returns'

FREQ = 52

In [30]:
info = pd.read_excel(INFILE,sheet_name=SHEET_INFO)
info.set_index('ticker',inplace=True)
info.loc[TICKS].rename(columns={
    'name': 'Name',
    'gics_sector_name': 'Sector',
    'mkt cap': 'Market capitalization',
})

,Name,Sector,Market capitalization
ticker,,,
AAPL,Apple Inc,Information Technology,4.631217e+12
NVDA,NVIDIA Corp,Information Technology,5.105232e+12
MSFT,Microsoft Corp,Information Technology,2.860690e+12
GOOGL,Alphabet Inc,Communication Services,4.335816e+12
AMZN,Amazon.com Inc,Consumer Discretionary,2.639149e+12
META,Meta Platforms Inc,Communication Services,1.698738e+12
TSLA,Tesla Inc,Consumer Discretionary,1.531434e+12
AVGO,Broadcom Inc,Information Technology,1.902889e+12
BRK/B,Berkshire Hathaway Inc,Financials,1.064448e+12


In [31]:
rets = pd.read_excel(INFILE,sheet_name=SHEET_RETURNS)
rets.set_index('date',inplace=True)
rets = rets[TICKS]

In [32]:
bench = pd.read_excel(INFILE,sheet_name=SHEET_BENCH)
bench.set_index('date',inplace=True)
rets[TICK_ETF] = bench[TICK_ETF]

***

# 1. Risk Statistics

### 1.1.

Display a table with the following metrics for each of the return series.

* mean (annualized)
* volatility (annualized)
* Sharpe ratio (annualized)
* skewness
* kurtosis
* maximum drawdown

#### Note
We  have total returns, and Sharpe ratio is technically defined for excess returns. Don't worry about the difference. (Or subtract `SHV` if you prefer.)

In [34]:
import numpy as np
ann_mean2 = rets.mean() * FREQ
ann_vol2 = rets.std() * np.sqrt(FREQ)
sharpe2 = ann_mean2 / ann_vol2
cum_rets2 = (1 + rets).cumprod()
rolling_max2 = cum_rets2.cummax()
drawdown2= (cum_rets2 - rolling_max2) / rolling_max2
mdd2 = drawdown2.min()
rets_skew = rets.skew()
rets_kurt = rets.kurtosis()
summary_stats2 = pd.DataFrame(
    {"annualized_mean": ann_mean2, 
     "annualized volatility": ann_vol2, 
     "sharpe ratio": sharpe2,
     "max drawdown": mdd2,
     "skew": rets_skew,
     "kurtosis": rets_kurt}
    )

print(summary_stats2)

       annualized_mean  annualized volatility  sharpe ratio  max drawdown  \
AAPL          0.248517               0.277602      0.895227     -0.346407   
NVDA          0.645347               0.454393      1.420241     -0.659360   
MSFT          0.235299               0.237816      0.989416     -0.350540   
GOOGL         0.269840               0.287311      0.939192     -0.418641   
AMZN          0.261284               0.304504      0.858063     -0.548307   
META          0.232050               0.357490      0.649108     -0.760252   
TSLA          0.437496               0.579650      0.754758     -0.722459   
AVGO          0.393854               0.382501      1.029681     -0.400358   
BRK/B         0.134847               0.188241      0.716352     -0.264770   
LLY           0.301432               0.298242      1.010699     -0.344750   
SPY           0.145162               0.169020      0.858844     -0.318291   

           skew  kurtosis  
AAPL  -0.182144  1.804677  
NVDA   0.350905  1.

### 1.2.

As a standalone investment, which is most attractive? And least? Justify your answer.

<!-- Rank by Sharpe ratio, since it’s return per unit of risk. Then use the tail stats to refine the answer. A stock with a high Sharpe but a very deep max drawdown, strong negative skew or high kurtosis is less attractive than its Sharpe alone suggests. Name the most and least attractive, and cite two or three numbers for each. -->

Looking at the output table, I see that NVDA has the highest sharpe ratio of 1.42 which tells me its return per unit of risk. NVDA also has a skew of 0.35 which is a positive number, so it tells me that NVDA's extreme returns are more often large gains than large losses. Additionally, NVDA's kurtosis is 1.49 which is relativley small to the other tickers which tells me that although NVDA still has extreme weeks happening more often than the normal distribution can predict, it happens less often than most of the other tickers. The least attractive investment is META with a sharpe ratio of 0.649, which is the lowest amongst these tickers so has the lease amount of return per unit of risk. Additionally, META has the second largest kurtosis and the largest max drawdown of 0.76. These statistics tell me that this ticker takes on a lot of risk and there's not enough return per unit of that risk, hence the low sharpe ratio. 

### 1.3.

For each investment, estimate a regression against `SPY`. Report the 
* alpha (annualized as a mean)
* beta
* info ratio
* r-squared

Based on this table, which investment seems most attractive relative to holding `SPY`?

In [ ]:
import statsmodels.api as sm
slope = sm.add_constant(rets[TICK_ETF])
data_table = pd.DataFrame(index=rets.columns.drop(TICK_ETF),
                    columns=["alpha", "beta", "info ratio", "r2"])
for col in data_table.index:
    ols_model = sm.OLS(rets[col], slope, missing="drop").fit()
    alpha = ols_model.params["const"] * FREQ
    resid_vol = ols_model.resid.std() * np.sqrt(FREQ)
    data_table.loc[col] = [alpha, ols_model.params[TICK_ETF], alpha / resid_vol, ols_model.rsquared]
print(data_table.astype(float))

          alpha      beta  info ratio        r2
AAPL   0.086496  1.116137    0.424725  0.461814
NVDA   0.395978  1.717868    1.132906  0.408313
MSFT   0.087785  1.016205    0.533694  0.521623
GOOGL  0.109599  1.103881    0.501628  0.421712
AMZN   0.100645  1.106619    0.418851  0.377300
META   0.060409  1.182411    0.203802  0.312525
TSLA   0.183613  1.748968    0.368251  0.260080
AVGO   0.187268  1.423137    0.629681  0.395462
BRK/B  0.021364  0.781767    0.159348  0.492723
LLY    0.213495  0.605791    0.762169  0.117865


Based on this table, I would say that the most attractive investment relative to holding SPY is NVDA. NVDA has the alpha which tells me that beyong NVDA's market exposure, NVDA earned an extra 39.6% a year that SPY cannot explain. This is further enforced by the high information rate of NVDA
highest information rate which tells me that the alpha is consistent and not as volatile as the other tickers. NVDA also has a high beta of 1.717 which tells me that as SPY moves 1%, NVDA also tends to move 1.72% in the same direction.

***

# 2. Portfolio Allocation

### 2.1.

Display the correlation matrix of the returns.

* Based on this information, which investment do you anticipate will get extra weight in the portfolio, beyond what it would merit for its mean return?

* Report the maximally correlated assets and the minimally correlated assets.

### 2.2.

Calculate the weights of the mean-variance optimized portfolio, also called the tangency portfolio.

* Display a table indexed by each investment, with the optimal weights in one column and the Sharpe ratios in another column.

* Do the investments with the best Sharpe ratios tend to get the biggest weights?

#### Note:
To estimate the optimal weights, consider using the provided function below.

In [110]:
def optimized_weights(returns,dropna=True,scale_cov=1):
    if dropna:
        returns = returns.dropna()

    covmat_full = returns.cov()
    covmat_diag = np.diag(np.diag(covmat_full))
    covmat = scale_cov * covmat_full + (1-scale_cov) * covmat_diag

    weights = np.linalg.solve(covmat,returns.mean())
    weights = weights / weights.sum()

    if returns.mean() @ weights < 0:
        weights = -weights

    return pd.DataFrame(weights, index=returns.columns)

### 2.3.

Report the following performance statistics of the portfolio achieved with the optimized weights calculated above.
* mean
* volatility
* Sharpe

(Annualize all three statistics.)

### 2.4.

Try dropping the asset which had the biggest short position from the investment set. Re-run the optimization. What do you think of these new weights compared to the original optimized weights?

What is going on?

****

# 3. EXTRA - Other Estimations

### 1.

In `Section 2`, we estimated the replications using an intercept. Try the full-sample estimation, but this time without an intercept.

$$\begin{aligned}
r^{\text{hfri}}_{t} &= \alpha^{\text{merr}} + x_{t}^{\text{merr}}\beta^{\text{merr}} + \epsilon_{t}^{\text{merr}}\\[5pt]
\check{r}^{\text{hfri}}_{t} &= \check{\alpha}^{\text{merr}} + x_{t}^{\text{merr}}\check{\beta}^{\text{merr}}
\end{aligned}$$

Report

* the regression beta. How does it compare to the estimated beta with an intercept, $\hat{\beta}^{\text{merr}}$?

* the mean of the fitted value, $\check{r}^{\text{hfri}}_{t}$. How does it compare to the mean of the HFRI?

* the correlations of the fitted values, $\check{r}^{\text{hfri}}_{t}$ to the HFRI. How does the correlation compare to that of the fitted values with an intercept, $\hat{r}^{\text{hfri}}_{t}$

Do you think Merrill and ProShares fit their replicators with an intercept or not?

### 2.

Merrill constrains the weights of each asset in its replication regression of HFRI. Try constraining your weights by re-doing 2.6.

* Use Non-Negative Least Squares (NNLS) instead of OLS.
* Go further by using a Generalized Linear Model to put separate interval constraints on each beta, rather than simply constraining them to be non-negative.

#### Hints
* Try using LinearRegression in scikit-learn with the parameter `positive=True`. 
* Try using GLM in statsmodels.

# 4. EXTRA - Other Decompositions

### 1. 

Let's decompose a few other targets to see if they behave as their name suggests.

* Regress HEFA on the same style factors used to decompose HFRI. Does HEFA appear to be a currency-hedged version of EFA?

* Decompose TRVCI with the same style factors used to decompose HFRI. The TRVCI Index tracks venture capital funds--in terms of our styles, what best describes venture capital?

* TAIL is an ETF that tracks SPY, but that also buys put options to protect against market downturns. Calculate the statistics in questions 2.1-2.3 for TAIL. Does it seem to behave as indicated by this description? That is, does it have high correlation to SPY while delivering lower tail risk?

### 2. 

The ProShares case introduces Levered ETFs. ProShares made much of its name originally through levered, or "geared" ETFs.

Explain conceptually why Levered ETFs may track their index well for a given day but diverge over time. How is this exacerbated in volatile periods like 2008?

### 3.

Analyze SPXU and UPRO relative to SPY.
- SPXU is ProShares -3x SPX ETF.
- UPRO is ProShres +3x SPX ETF.

Questions:
* Analyze them with the statistics from 2.1-2.3. 

* Do these two ETFs seem to live up to their names?

* Plot the cumulative returns of both these ETFs along with SPY.

* What do you conclude about levered ETFs?

***